In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# -------------------------
# 0) Daten laden
# -------------------------
df = pd.read_csv("survey_results_cleaned_final.csv")
print("Loaded:", df.shape)

# Bool-Spalten ggf. in int umwandeln
bool_cols = df.select_dtypes(include=["bool"]).columns
for col in bool_cols:
    df[col] = df[col].astype(int)

target_col = "Employment"

# Target muss vorhanden sein
df = df.dropna(subset=[target_col]).copy()

# ✅ NEU: seltene/unnütze Klasse droppen (verhindert 1-sample Klassen im Test)
df = df[df[target_col] != "i prefer not to say"].copy()

# Optional: "Other" extrem selten -> droppen (bei dir waren es 2)
df = df[df[target_col] != "Other"].copy()

print("\nTarget distribution:")
print(df[target_col].value_counts())

# -------------------------
# 1) Feature-Spalten bestimmen
# -------------------------
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols = [c for c in text_cols if c not in {"cluster", target_col}]  # target nicht in Text
df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in {target_col, "cluster"}]

# Optional: ID raus, falls vorhanden
for maybe_id in ["ResponseId"]:
    if maybe_id in num_cols:
        num_cols.remove(maybe_id)

print("\nTextspalten:", len(text_cols))
print("Numerische Spalten:", len(num_cols))

# ✅ NEU: Numerische NaNs droppen (oder imputer verwenden)
df = df.dropna(subset=num_cols).copy()

X = df[["__text__"] + num_cols].copy()
y = df[target_col].astype(str).copy()

print("\nX shape:", X.shape)
print("y shape:", y.shape)

# -------------------------
# 2) Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain size:", len(X_train))
print("Test size:", len(X_test))
print("Train dist:\n", y_train.value_counts(normalize=True))
print("Test dist:\n", y_test.value_counts(normalize=True))

# -------------------------
# 3) Preprocessing
# -------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=5,            # ✅ NEU (vorher 2)
            max_df=0.9,
            max_features=20000,  # ✅ NEU
            sublinear_tf=True    # ✅ NEU
        ), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

# -------------------------
# 4) Pipeline
# -------------------------
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(
        LinearSVC(penalty="l1", dual=False, C=0.5, max_iter=20000)
    )),
    ("classifier", LinearSVC(max_iter=20000))
])

# -------------------------
# 5) GridSearch
# -------------------------
parameters = {
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__min_df": [5, 10],      # ✅ NEU: stabilere Varianten
    "preprocessing__text__max_df": [0.9],

    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(
    pipeline,
    param_grid=parameters,
    scoring="f1_macro",
    verbose=2,
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBeste Performance (CV, f1_macro):", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

# -------------------------
# 6) Evaluation
# -------------------------
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix (rows=true, cols=pred):")
labels_sorted = sorted(y.unique())
print(confusion_matrix(y_test, y_pred, labels=labels_sorted))

print("\nClassification Report (Train):")
y_pred_train = best_model.predict(X_train)
print(classification_report(y_train, y_pred_train, zero_division=0))


Loaded: (20603, 40)

Target distribution:
Employment
employed                                                16257
independent contractor, freelancer, or self-employed     2709
student                                                   995
not employed                                              594
retired                                                     2
Name: count, dtype: int64

Textspalten: 30
Numerische Spalten: 8

X shape: (12982, 9)
y shape: (12982,)

Train size: 10385
Test size: 2597
Train dist:
 Employment
employed                                                0.850650
independent contractor, freelancer, or self-employed    0.116418
not employed                                            0.018199
student                                                 0.014733
Name: proportion, dtype: float64
Test dist:
 Employment
employed                                                0.850597
independent contractor, freelancer, or self-employed    0.116673
not employed                

In [9]:
# %% md
# XGBoost + OneHot-Encoding – JobSat (Low/Medium/High)
# Ziel: JobSat klassifizieren (3 Klassen)
# Features: Numerik + kategoriale Spalten via OneHotEncoder
# Modell: XGBClassifier

# %%
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

from xgboost import XGBClassifier

# %%
# 1) Daten laden
df = pd.read_csv("survey_results_cleaned_final.csv")
print("Loaded:", df.shape)

# Optional: bool -> int (falls vorhanden)
bool_cols = df.select_dtypes(include=["bool"]).columns
for c in bool_cols:
    df[c] = df[c].astype(int)

# %%
# 2) Target: JobSat -> Klassen (Low/Medium/High)
target_col = "JobSat"

# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=[target_col]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return 0   # Low
    elif x <= 6:
        return 1   # Medium
    else:
        return 2   # High

y = df[target_col].apply(map_jobsat).astype(int)

print("Target distribution:")
print(y.value_counts().sort_index())

# %%
# 3) Feature-Spalten bestimmen
# -> Numerik: int/float (ohne JobSat, ohne cluster falls vorhanden)
# -> Kategorial: object (Strings), ebenfalls ohne cluster

drop_cols = {target_col}
if "cluster" in df.columns:
    drop_cols.add("cluster")

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64","float64","int32","float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in drop_cols]

# Kategoriale Spalten
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c not in drop_cols]

# Optional: ID-Spalten raus (falls vorhanden)
for maybe_id in ["ResponseId"]:
    if maybe_id in num_cols: num_cols.remove(maybe_id)
    if maybe_id in cat_cols: cat_cols.remove(maybe_id)

print("Numeric cols:", len(num_cols))
print("Categorical cols:", len(cat_cols))

# %%
# 4) Gemeinsamer Clean-Step: nur Reihen behalten, wo Features + Target vollständig sind
# Für OneHot ist NaN ok (Encoder kann mit fehlenden umgehen, wenn wir sie als Kategorie behandeln),
# ABER XGBoost + sklearn Pipeline ist am stabilsten, wenn wir NaNs in cat als "MISSING" füllen.
X = df[num_cols + cat_cols].copy()

# Kategoriale NaNs füllen
for c in cat_cols:
    X[c] = X[c].fillna("MISSING").astype(str)

# Numerische NaNs: XGBoost kann NaNs grundsätzlich, aber damit train/test konsistent ist,
# lassen wir sie drin. Wenn du willst, kannst du hier auch dropna machen:
# X = X.dropna(subset=num_cols)
# y = y.loc[X.index]

# Wichtig: Index syncen
y = y.loc[X.index]
assert len(X) == len(y), f"Mismatch X={len(X)} vs y={len(y)}"

print("X shape:", X.shape, "| y shape:", y.shape)

# %%
# 5) Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# %%
# 6) Preprocessing: OneHotEncoder für Kategorien
#    Wichtig: seltene Kategorien bündeln -> verhindert Feature-Explosion.
#    min_frequency= z.B. 50 heißt: Kategorien die <50 mal vorkommen, werden "infrequent".
#    (Wenn sklearn zu alt ist und min_frequency nicht unterstützt -> sag Bescheid, ich gebe Fallback.)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=50,               # <<< ggf. anpassen (25 / 50 / 100)
            sparse_output=True
        ), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

# %%
# 7) Modell: XGBoost Multiclass
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

model = Pipeline([
    ("preprocess", preprocessor),
    ("clf", xgb)
])

# %%
# 8) Trainieren
model.fit(X_train, y_train)

# %%
# 9) Evaluation
y_pred = model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred, labels=[0,1,2], target_names=["Low","Medium","High"]))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0,1,2]))

# Optional: Train-Report (Overfitting-Check)
y_pred_train = model.predict(X_train)
print("\nClassification Report (Train):")
print(classification_report(y_train, y_pred_train, labels=[0,1,2], target_names=["Low","Medium","High"]))


Loaded: (20603, 40)
Target distribution:
JobSat
0     1061
1     3935
2    12511
Name: count, dtype: int64
Numeric cols: 7
Categorical cols: 31
X shape: (17507, 38) | y shape: (17507,)
Train size: 14005
Test size: 3502
Classification Report (Test):
              precision    recall  f1-score   support

         Low       0.33      0.05      0.09       212
      Medium       0.42      0.13      0.20       787
        High       0.74      0.96      0.84      2503

    accuracy                           0.72      3502
   macro avg       0.50      0.38      0.38      3502
weighted avg       0.65      0.72      0.65      3502

Confusion Matrix (rows=true, cols=pred):
[[  11   44  157]
 [  14  105  668]
 [   8   99 2396]]

Classification Report (Train):
              precision    recall  f1-score   support

         Low       0.99      0.41      0.58       849
      Medium       0.90      0.39      0.54      3148
        High       0.81      0.99      0.89     10008

    accuracy            